In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
df = pd.read_csv(r"C:\Users\dell\Downloads\movies_data.csv")
df.head()

,Name,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3
0,#Gadhvi (He thought he was Gandhi),2019,109,Drama,7.0,8,Gaurav Bakshi,Rasika Dugal,Vivek Ghamande,Arvind Jangid
1,#Yaaram,2019,110,"Comedy, Romance",4.4,35,Ovais Khan,Prateik,Ishita Raj,Siddhant Kapoor
2,...Aur Pyaar Ho Gaya,1997,147,"Comedy, Drama, Musical",4.7,827,Rahul Rawail,Bobby Deol,Aishwarya Rai Bachchan,Shammi Kapoor
3,...Yahaan,2005,142,"Drama, Romance, War",7.4,1086,Shoojit Sircar,Jimmy Sheirgill,Minissha Lamba,Yashpal Sharma
4,?: A Question Mark,2012,82,"Horror, Mystery, Thriller",5.6,326,Allyson Patel,Yash Dave,Muntazir Ahmad,Kiran Bhatia


In [29]:
df.isnull().sum()

Name        0
Year        0
Duration    0
Genre       0
Rating      0
Votes       0
Director    0
Actor 1     0
Actor 2     0
Actor 3     0
dtype: int64

In [30]:
transactions = df['Genre'].str.split(r"\s*,\s*").tolist()

In [31]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary,columns= te.columns_)
df_encoded.head()

,Action,Adventure,Animation,Biography,Comedy,Crime,Documentary,Drama,Family,Fantasy,...,Music,Musical,Mystery,News,Romance,Sci-Fi,Sport,Thriller,War,Western
0,False,False,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,True,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
2,False,False,False,False,True,False,False,True,False,False,...,False,True,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,True,False,False,...,False,False,False,False,True,False,False,False,True,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,True,False,False


In [32]:
frequent_itemsets = apriori(df_encoded, min_support=0.05, use_colnames=True)

# Step 4: Association Rules generate karein (min_confidence = 30%)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.3)

# Step 5: Top rules ko Lift ke hisab se sort karke dekhein
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values(by='lift', ascending=False).head(10)

,antecedents,consequents,support,confidence,lift
9,"frozenset({Action, Drama})",frozenset({Crime}),0.058844,0.335347,2.168836
10,frozenset({Crime}),"frozenset({Action, Drama})",0.058844,0.380571,2.168836
1,frozenset({Action}),frozenset({Crime}),0.090299,0.303084,1.960176
0,frozenset({Crime}),frozenset({Action}),0.090299,0.584000,1.960176
8,"frozenset({Crime, Drama})",frozenset({Action}),0.058844,0.568259,1.907343
5,frozenset({Family}),frozenset({Drama}),0.058844,0.800481,1.193341
4,frozenset({Crime}),frozenset({Drama}),0.103552,0.669714,0.998397
7,"frozenset({Crime, Action})",frozenset({Drama}),0.058844,0.651663,0.971487
6,frozenset({Romance}),frozenset({Drama}),0.158332,0.649275,0.967927
2,frozenset({Action}),frozenset({Drama}),0.175473,0.588968,0.878022


In [33]:
#drop the row where city is mumbai
#df.drop(df['city']df['city'] =='mumbai'].index,insplace=True)

In [34]:
df_group_2 = df.groupby(['Director','Actor 1','Actor 2','Actor 3'])['Name'].apply(lambda x: list(set(x)))

In [35]:
pd.DataFrame(df_group_2)

Name
Director     Actor 1            Actor 2             Actor 3                                  
A. Bhimsingh Ashok Kumar        Mala Sinha          Dharmendra               [Pooja Ke Phool]
                                Sulochana Latkar    Sunil Dutt                      [Mehrban]
                                Sunil Dutt          Nutan                        [Bhai Bahen]
                                Waheeda Rehman      Pradeep Kumar                     [Rakhi]
             Christopher Coelho Chandramohan        Rajasulochana              [Karunamayudu]
...                                                                                       ...
Zoya Akhtar  Farhan Akhtar      Konkona Sen Sharma  Rishi Kapoor             [Luck by Chance]
             Hrithik Roshan     Farhan Akhtar       Abhay Deol     [Zindagi Na Milegi Dobara]
             Vijay Varma        Nakul Roshan Sahdev Ranveer Singh                 [Gully Boy]
Zubair Khan  Vicky Ahuja        Saaikat Bagbaan     Ajaz Khan              [Lakeer Ka Fakeer]
Zunaid Memon Nassar Abdulla     Ravi Baswani        Rakesh Bedi                    [The Film]

[5641 rows x 1 columns]

In [36]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
import ast

In [37]:
df1 = pd.read_csv(r"C:\Users\dell\Downloads\movies.csv").dropna()
df1.head()

,Unnamed: 0,URLs,Year_Released,Title,IMDB Ids,Cast,Plot_Summary,Review,Genre,Released Date,Plot Keywords,Duration,Age of content
0,0,https://www.imdb.com//title/tt1187043/?ref_=fn...,2009.0,3 Idiots,tt1187043,"With Aamir Khan, Madhavan, Sharman Joshi, Kare...","""Two friends are searching for their long lost...","""I was lucky to see the first day show of 3 id...","[""Comedy"",""Drama""]","""2009-12-25""","""papadum,hairy chest,against the system,coming...","""2H50M""",13.0
1,1,https://www.imdb.com//title/tt0986264/?ref_=fn...,2007.0,Taare Zameen Par,tt0986264,"With Darsheel Safary, Aamir Khan, Tisca Chopra...","""An eight-year-old boy is thought to be a lazy...","""(I notice that there are many spoilers here a...","[""Drama"",""Family""]","""2007-12-21""","""teacher student relationship,dyslexia,against...","""2H45M""",15.0
2,2,https://www.imdb.com//title/tt2338151/?ref_=fn...,2014.0,PK,tt2338151,"With Aamir Khan, Rajinder Sharma Nanu, Anushka...","""An alien on Earth loses the only device he ca...","""First half of the movie focuses on comedic as...","[""Comedy"",""Drama"",""Sci-Fi""]","""2014-12-19""","""guru,religion,tv reporter,alien on earth,papa...","""2H33M""",8.0
3,3,https://www.imdb.com//title/tt5074352/?ref_=fn...,2016.0,Dangal,tt5074352,"With Aamir Khan, Fatima Sana Shaikh, Sanya Mal...","""Former wrestler Mahavir Singh Phogat and his ...","""If I would I&apos;ve seen this movie in 2016,...","[""Action"",""Biography"",""Drama""]","""2016-12-23""","""small town,father daughter relationship,male ...","""2H41M""",6.0
4,4,https://www.imdb.com//title/tt0405508/?ref_=fn...,2006.0,Rang De Basanti,tt0405508,"With Aamir Khan, Siddharth, Sharman Joshi, Kun...","""The story of six young Indians who assist an ...","""Paint It Yellow . Oops .that&apos;s suppose t...","[""Comedy"",""Crime"",""Drama""]","""2006-01-26""","""delhi india,murder,friend,india,awakening""","""2H47M""",16.0


In [38]:
df1.isnull().sum()

Unnamed: 0        0
URLs              0
Year_Released     0
Title             0
IMDB Ids          0
Cast              0
Plot_Summary      0
Review            0
Genre             0
Released Date     0
Plot Keywords     0
Duration          0
Age of content    0
dtype: int64

In [39]:
df_movies = df1[['Title','Genre']]


In [40]:
df_movies

,Title,Genre
0,3 Idiots,"[""Comedy"",""Drama""]"
1,Taare Zameen Par,"[""Drama"",""Family""]"
2,PK,"[""Comedy"",""Drama"",""Sci-Fi""]"
3,Dangal,"[""Action"",""Biography"",""Drama""]"
4,Rang De Basanti,"[""Comedy"",""Crime"",""Drama""]"
...,...,...
3118,Mehmaan,"[""Short"",""Drama""]"
3119,Sorry Daddy,"[""Action""]"
3120,Raadha Aur Seeta,"[""Drama"",""Family""]"
3121,IPC Section 302,


In [41]:
df_movies['Genre'] = df_movies['Genre'].map(lambda val: np.nan if val == " " else val)

In [42]:
df_movies.info()

<class 'pandas.DataFrame'>
Index: 2964 entries, 0 to 3122
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Title   2964 non-null   str  
 1   Genre   2361 non-null   str  
dtypes: str(2)
memory usage: 163.6 KB


In [43]:
df_movies["Genre"][0]

'["Comedy","Drama"]'

In [44]:
def convert_into_list(value):
    if isinstance(value,str):
        return ast.literal_eval(value)
    return value
        

In [45]:
df_movies["Genre"] = df_movies['Genre'].apply(convert_into_list)

In [46]:
df_movies

,Title,Genre
0,3 Idiots,"[Comedy, Drama]"
1,Taare Zameen Par,"[Drama, Family]"
2,PK,"[Comedy, Drama, Sci-Fi]"
3,Dangal,"[Action, Biography, Drama]"
4,Rang De Basanti,"[Comedy, Crime, Drama]"
...,...,...
3118,Mehmaan,"[Short, Drama]"
3119,Sorry Daddy,[Action]
3120,Raadha Aur Seeta,"[Drama, Family]"
3121,IPC Section 302,NaN


In [47]:
df_movies.isnull().sum()

Title      0
Genre    603
dtype: int64

In [48]:
df_movies = df_movies.dropna()

In [49]:
encode  = MultiLabelBinarizer()


In [50]:
df_movies.index = df_movies['Title']

In [51]:
df_movies.head(4)

,Title,Genre
Title,,
3 Idiots,3 Idiots,"[Comedy, Drama]"
Taare Zameen Par,Taare Zameen Par,"[Drama, Family]"
PK,PK,"[Comedy, Drama, Sci-Fi]"
Dangal,Dangal,"[Action, Biography, Drama]"


In [52]:
encode_data= encode.fit_transform(df_movies['Genre'])
df_encoded = pd.DataFrame(encode_data,index= df_movies.index,columns= encode.classes_)
df_encoded.head()

,Action,Adventure,Animation,Biography,Comedy,Crime,Documentary,Drama,Family,Fantasy,...,News,Reality-TV,Romance,Sci-Fi,Short,Sport,Talk-Show,Thriller,War,Western
Title,,,,,,,,,,,,,,,,,,,,,
3 Idiots,0,0,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
Taare Zameen Par,0,0,0,0,0,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
PK,0,0,0,0,1,0,0,1,0,0,...,0,0,0,1,0,0,0,0,0,0
Dangal,1,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
Rang De Basanti,0,0,0,0,1,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [60]:
df_encoded.index = df_encoded.index.str.lower().str.strip()

In [61]:
df_encoded.index

Index(['3 idiots', 'taare zameen par', 'pk', 'dangal', 'rang de basanti',
       'lagaan: once upon a time in india', 'my name is khan',
       'swades: we, the people', 'gangs of wasseypur', 'gangs of wasseypur',
       ...
       'haseenon ka devata', 'mazloom', 'mere genie uncle', 'kaanch ki deewar',
       'woh jo hasina', 'swarg jaisaa ghar', 'mehmaan', 'sorry daddy',
       'raadha aur seeta', 'calapor'],
      dtype='str', name='Title', length=2361)

In [ ]:
def recomneded